# Spectral Band Explorer — StreamCI / waterbear (live)

Pulls the current contents of the `waterbear` collection from the StreamCI API into a DataFrame, then renders interactive time-series plots for each spectral band (`band_r` … `band_w`).

**Notes on the data:**
- `time_h` is stuck at the 1970 epoch in this collection, so `published_at` is used as the timestamp.
- Rows where all six bands are missing (bare heartbeat records) are dropped.
- Each plot is colored by `device_id`, has a range slider for zooming, and supports click-to-hide devices in the legend (double-click a legend entry to isolate it).

Requires: `requests`, `pandas`, `plotly`.

In [ ]:
import requests
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BANDS = ["band_r", "band_s", "band_t", "band_u", "band_v", "band_w"]

TARGET = "waterbear"
SECRET_KEY = "uRSGDmkla+B*e3MFoiQ9#oSZ2tOzn3j5"


def fetch(query, target, secret_key):
    headers = {"Content-type": "application/json"}
    data = {
        "auth": {
            "target": target,
            "authtype": "secret",
            "secret_key": secret_key,
        },
        "request": query,
    }
    addr = "https://api.streamci.org/query"
    res = requests.post(addr, data=json.dumps(data), headers=headers)
    res.raise_for_status()
    # Response is NDJSON: one JSON object per line
    lines = res.text.strip().splitlines()
    return [json.loads(line) for line in lines]

In [ ]:
# Pull live data
query = {
    "method": "query",
    "query": {},
    # "projection": ["value", "sequence", "timestamp"],
    # "sort": {"time_h": 1},
}

results = fetch(query, TARGET, SECRET_KEY)
print("records fetched:", len(results))

raw = pd.DataFrame(results)
raw.head(2)

In [ ]:
# Clean into the plotting DataFrame
df = raw.copy()

# published_at is the only usable timestamp (time_h is all 1970-01-01)
# (no format= arg: format="ISO8601" is pandas 2.x-only; plain parsing works on 1.x and 2.x)
df["timestamp"] = pd.to_datetime(df["published_at"], utc=True)

# Make sure all band columns exist and are numeric, even if some records lack them
for band in BANDS:
    if band not in df.columns:
        df[band] = pd.NA
    df[band] = pd.to_numeric(df[band], errors="coerce")

# Drop rows with no band data at all (heartbeat-only records)
df = df.dropna(subset=BANDS, how="all").copy()

# Normalize missing device IDs so they still show up in plots
df["device_id"] = df["device_id"].fillna("(no device_id)")

df = df.sort_values("timestamp").reset_index(drop=True)

print(f"{len(df)} records, {df['device_id'].nunique()} devices")
print(f"Time range: {df['timestamp'].min()}  ->  {df['timestamp'].max()}")
df[["timestamp", "device_id"] + BANDS].head()

In [ ]:
# Records per device
df["device_id"].value_counts().to_frame("n_records")

In [ ]:
def plot_band(band, devices=None, log_y=False, markers=True):
    """Interactive time series for one band.

    Parameters
    ----------
    band : str            e.g. "band_r"
    devices : list[str]   optional subset of device_ids to plot
    log_y : bool          log-scale the y axis (useful given the huge dynamic range)
    markers : bool        show point markers on the lines
    """
    d = df if devices is None else df[df["device_id"].isin(devices)]
    fig = px.line(
        d.dropna(subset=[band]),
        x="timestamp",
        y=band,
        color="device_id",
        markers=markers,
        log_y=log_y,
        title=f"{band} over time",
        labels={"timestamp": "Time (UTC)", band: f"{band} (counts)", "device_id": "Device"},
    )
    fig.update_traces(marker=dict(size=4), line=dict(width=1))
    fig.update_layout(
        hovermode="x unified",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        xaxis=dict(rangeslider=dict(visible=True)),
        title_y=0.98,
    )
    return fig

In [ ]:
plot_band("band_r").show()

In [ ]:
plot_band("band_s").show()

In [ ]:
plot_band("band_t").show()

In [ ]:
plot_band("band_u").show()

In [ ]:
plot_band("band_v").show()

In [ ]:
plot_band("band_w").show()

## All bands for a single device

Use the dropdown to switch devices and compare the six bands together — handy for spotting spectral shape changes (e.g. NDVI-relevant red vs. NIR divergence) on one sensor.

In [ ]:
devices = df["device_id"].value_counts().index.tolist()

fig = go.Figure()
n_bands = len(BANDS)

for i, dev in enumerate(devices):
    d = df[df["device_id"] == dev]
    for band in BANDS:
        fig.add_trace(go.Scatter(
            x=d["timestamp"], y=d[band],
            mode="lines+markers", name=band,
            marker=dict(size=4), line=dict(width=1),
            visible=(i == 0),
        ))

buttons = []
for i, dev in enumerate(devices):
    vis = [False] * (len(devices) * n_bands)
    vis[i * n_bands:(i + 1) * n_bands] = [True] * n_bands
    buttons.append(dict(label=dev, method="update",
                        args=[{"visible": vis},
                              {"title": f"All bands — {dev}"}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.15)],
    title=f"All bands — {devices[0]}",
    xaxis_title="Time (UTC)", yaxis_title="Counts",
    hovermode="x unified", height=550,
    xaxis=dict(rangeslider=dict(visible=True)),
)
fig.show()

## Tips

- **Refresh**: re-run the fetch cell and everything below it to pull the latest data.
- **Log scale**: values span 0 to ~60,000 with lots of near-zero nighttime readings — try `plot_band("band_r", log_y=True)`.
- **Subset devices**: `plot_band("band_v", devices=["Test_TT", "HortPark_Open"])`
- **Lines only** (faster for dense data): `plot_band("band_w", markers=False)`
- **Server-side filtering**: to fetch less, put a Mongo-style filter in the query, e.g. `query["query"] = {"device_id": "Test_TT"}`.